In [ ]:
# ---
# Convert Official Gene Symbols to PmUG01 IDs using NCBI Entrez API
# Target: Genes like ACBC4 (not XM_/XR_)
# Output: Updated DataFrame with "Converted_ID" as PmUG01_xxxxxx
# Batches queries and saves after each batch
# ---


In [ ]:

import pandas as pd
import time
from Bio import Entrez

# Configuration
Entrez.email = "halsal4u@gmail.com"  # NCBI requires a valid email
input_file = "analysis_results/variants_analysis/files/cleaned_extracted_variants.tsv"
output_file = "analysis_results/variants_analysis/files/PmUGO_ID_converted_genes.csv"
BATCH_SIZE = 10

# Load & Filter Data
df = pd.read_csv(input_file, sep="\t")
df.columns = ["Gene", "Converted_ID"]  # Ensure correct headers

# Only keep gene symbols (not XM_/XR_) that haven't been converted yet
genes_to_convert = [
    row["Gene"]
    for _, row in df.iterrows()
    if isinstance(row["Gene"], str) and not row["Gene"].startswith(("XM_", "XR_")) and
    (pd.isna(row["Converted_ID"]) or not str(row["Converted_ID"]).startswith("PmUG01"))
]

# Fetch PmUG01 ID
def fetch_locus_tag(gene_symbol):
    """Returns the PmUG01 locus tag given an official gene symbol (e.g., ACBC4)."""
    try:
        query = f"{gene_symbol} AND Plasmodium malariae[Organism]"
        print(f"🔍 Searching NCBI for: {query}")

        handle = Entrez.esearch(db="gene", term=query, retmode="xml")
        record = Entrez.read(handle)
        handle.close()

        if not record["IdList"]:
            print(f"⚠ No match found for {gene_symbol}")
            return "Not Found"

        gene_id = record["IdList"][0]

        handle = Entrez.efetch(db="gene", id=gene_id, retmode="xml")
        gene_data = Entrez.read(handle)
        handle.close()

        locus_tag = gene_data[0]["Entrezgene_gene"]["Gene-ref"].get("Gene-ref_locus-tag", "Not Found")
        print(f"✅ {gene_symbol} → {locus_tag}")
        return locus_tag

    except KeyError:
        print(f"⚠ Missing tag for {gene_symbol}")
    except Exception as e:
        print(f"❌ Error fetching {gene_symbol}: {e}")

    return "Not Found"

# Batch Process & Save
converted_results = {}

for i in range(0, len(genes_to_convert), BATCH_SIZE):
    batch = genes_to_convert[i:i + BATCH_SIZE]
    print(f"\n🔄 Processing batch {i // BATCH_SIZE + 1}/{-(-len(genes_to_convert) // BATCH_SIZE)}...")

    for gene_symbol in batch:
        converted_results[gene_symbol] = fetch_locus_tag(gene_symbol)
        time.sleep(0.5)  # Respect NCBI rate limits

    # Apply conversion to DataFrame
    df["Converted_ID"] = df.apply(
        lambda row: converted_results.get(row["Gene"], row["Converted_ID"]),
        axis=1
    )

    df.to_csv(output_file, index=False)
    print("✅ Progress saved.")

# Completion
print(f"\n🎉 Conversion complete! Results saved to '{output_file}'")
